# 02b — Package and Publish the Prepared Data to Kaggle

## Objective
Publish only the verified local outputs of notebooks 01 and 02. Both the data and
the updated code bundle remain **public** under the configured account. This notebook
does not launch a kernel or consume GPU training quota.

## Publication contract
1. Re-check split locks, all image/label hashes and observed overlap constraints.
2. Package approved images/labels, manifests, source attribution and the data card.
3. Upload the data dataset and verify its remote version, readiness and public visibility.
4. Update the code dataset for the current project layout and executed data notebooks.

Raw archives, unannotated/quarantined photos, personal pilot photos and credentials
are excluded from the upload. Upload receipts bind the remote version to a local SHA-256.

## 1. Load and revalidate the prepared dataset

In [1]:
from pathlib import Path
import json
import sys
import platform
import pandas as pd
from IPython.display import display

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "src/configs/sources.json").is_file())
REPORT = ROOT / "docs/reports/data_preparation"
REPORT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 85)
print("Python:", platform.python_version())
print("Project:", ROOT)

from dentalselfie.research import validate_prepared

manifests = pd.concat([pd.read_csv(ROOT / "data/manifests" / f"{task}.splits.csv", keep_default_na=False)
                       for task in ("caries", "gum")], ignore_index=True)
pairs = pd.read_csv(REPORT / "duplicate_candidates.csv", keep_default_na=False)
checks = validate_prepared(ROOT, manifests, pairs)
display(checks.groupby("task").agg(checked_classes=("class_id", "size"), status=("validation", "first")))
display(manifests.groupby(["task", "split"]).size().rename("images").reset_index())


Python: 3.11.15
Project: C:\Users\huynh\Desktop\oral-disease


,checked_classes,status
task,,
caries,3,passed
gum,9,passed


,task,split,images
0,caries,test,326
1,caries,train,1522
2,caries,val,327
3,gum,test,160
4,gum,train,747
5,gum,val,160


## 2. Inspect packaging and upload implementation

These are the actual shared implementations used by the cells below. Uploads are idempotent when the local artifact and recorded remote version match.

In [2]:
import inspect
from IPython.display import Code
from dentalselfie.package import package_data, package_code
from dentalselfie.publishing import publish_stage, verify_remote_data

display(Code(inspect.getsource(package_data), language="python"))
display(Code(inspect.getsource(publish_stage), language="python"))
display(Code(inspect.getsource(verify_remote_data), language="python"))


def package_data(root: Path) -> Path:
    config = json.loads((root / "src/configs/kaggle.json").read_text(encoding="utf-8"))
    target = root / ".venv/work/kaggle/staging/data"
    members = []
    # Verify BOTH tasks and image/label hashes before creating any package.
    for task in ("caries", "gum"):
        lock = root / "data/manifests" / f"{task}.lock.json"
        splits = root / "data/manifests" / f"{task}.splits.csv"
        if not lock.is_file() or not splits.is_file():
            raise ValueError(f"{task}: audited, locked splits required before upload")
        record = json.loads(lock.read_text(encoding="utf-8"))
        if record["splits_sha256"] != digest(splits):
            raise ValueError(f"{task}: split changed after lock")
        if not (root / "data/processed" / task / "dataset/data.yaml").is_file():
            raise ValueError(f"{task}: materialized dataset missing")
        dataset = root / "data/processed" / task / "dataset"
        with splits.open(encoding="utf-8", newline="") as stream:
            for row in csv.DictReader(stream):
                matches = list((dataset / "images" / row["split"]).glob(row["sample_id"] + ".*"))
                label = dataset / "labels" / row["split"] / f"{row['sample_id']}.txt"
                if len(matches) != 1 or digest(matches[0]) != row["sha256"] or digest(label) != row["label_sha256"]:
                    raise ValueError(f"{task}: prepared image/label differs from normalized manifest")
                members.extend([(matches[0], f"{task}/images/{row['split']}/{matches[0].name}"),
                                (label, f"{task}/labels/{row['split']}/{label.name}")])
        members.extend([(dataset / "data.yaml", f"{task}/data.yaml"),
                        (lock, f"manifests/{task}.lock.json"), (splits, f"manifests/{task}.splits.csv")])
    card = root / "docs/reports/data_preparation/DATA_CARD.md"
    if card.is_file():
        members.append((card, "DATA_CARD.md"))
    signature_rows = [(name, digest(source), source.stat().st_size) for source, name in members]
    signature_rows.append(("sources.json", digest(root / "src/configs/sources.json"), 0))
    signature = hashlib.sha256(json.dumps(signature_rows, sort_keys=True).encode()).hexdigest()
    receipt_path = target / "package_receipt.json"
    if receipt_path.is_file():
        receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
        if receipt["input_signature"] != signature:
            raise ValueError("Staged data belongs to another protocol; use an explicit new dataset version")
        if digest(target / "prepared.zip") != receipt["archive_sha256"]:
            raise ValueError("Staged archive checksum changed")
        return target
    target.mkdir(parents=True, exist_ok=True)
    partial = target / "prepared.zip.part"
    with zipfile.ZipFile(partial, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
        for source, name in members:
            archive.write(source, name)
        archive.write(root / "src/configs/sources.json", "sources.json")
    with zipfile.ZipFile(partial) as archive:
        if archive.testzip() is not None:
            raise ValueError("Prepared ZIP failed its CRC integrity check")
    partial.replace(target / "prepared.zip")
    shutil.copy2(root / "src/configs/sources.json", target / "sources.json")
    metadata = {"title": "DentalSelfie Data v1", "id": f"{config['owner']}/{config['data_slug']}",
                "licenses": [{"name": "CC-BY-4.0"}]}
    (target / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    description = (
        "Derived research datasets, CC BY 4.0. Original sources:\n"
        "https://doi.org/10.5281/zenodo.14827784\nhttps://doi.org/10.17632/3253gj88rr.1\n"
        "Changes: class remapping, format normalization and new group-aware splits.\n"
        "See sources.json and manifests for provenance. No DentalSelfie pilot photos included.\n"
    )
    if card.is_f

def publish_stage(root: Path, kind: str, stage: Path) -> dict:
    """Create/version an owned dataset under the explicitly configured visibility."""
    from kaggle.api.kaggle_api_extended import KaggleApi

    if kind not in ("data", "code", "roi", "results"):
        raise ValueError("Unknown publication kind")
    config = json.loads((root / "src/configs/kaggle.json").read_text())
    stage = stage.resolve()
    expected = (root / ".venv/work/kaggle/staging" / kind).resolve()
    if stage != expected or not stage.is_relative_to(root.resolve()):
        raise ValueError("Unexpected upload folder")
    metadata = json.loads((stage / "dataset-metadata.json").read_text())
    dataset = f"{config['owner']}/{config[kind + '_slug']}"
    if metadata["id"] != dataset:
        raise ValueError("Upload metadata points to a different account/dataset")
    artifact = stage / {"data": "prepared.zip", "code": "project.zip", "roi": "roi.zip", "results": "results.zip"}[kind]
    artifact_sha = digest(artifact)
    api = KaggleApi()
    api.authenticate()  # Uses the existing local credential configuration; never print it.
    existing = next((item for item in api.dataset_list(mine=True, search=config[kind + "_slug"])
                     if item.ref == dataset), None)
    expected_private = bool(config.get("private", True))
    if existing is not None and bool(existing.is_private) != expected_private:
        raise RuntimeError("Existing visibility differs from project policy; update and verify it explicitly first")
    receipt_path = root / "docs/reports/data_preparation/kaggle_upload_receipt.json"
    receipts = json.loads(receipt_path.read_text()) if receipt_path.is_file() else {}
    previous = receipts.get(kind)
    if (existing is not None and previous and previous["artifact_sha256"] == artifact_sha
            and previous["version"] == existing.current_version_number):
        previous["private"] = expected_private
        receipts[kind] = previous
        write_json(receipt_path, receipts)
        print(f"{kind}: identical dataset version already uploaded; visibility verified", flush=True)
        return previous
    cli = shutil.which("kaggle")
    if cli is None:
        raise RuntimeError("Kaggle CLI is unavailable")
    mode = "version" if existing is not None else "create"
    expected_version = int(existing.current_version_number) + 1 if existing is not None else 1
    command = [cli, "datasets", mode, "-p", str(stage), "-q"]
    if mode == "version":
        command += ["-m", "Verified DentalSelfie notebook workflow and artifacts"]
    elif not expected_private:
        command.append("--public")
    log_dir = root / ".venv/work/kaggle/upload_logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{kind}_{int(time.time())}.log"
    print(f"Uploading {kind}: {artifact.stat().st_size / 1_000_000:.1f} MB to {dataset} ({'private' if expected_private else 'public'})", flush=True)
    started = time.monotonic()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT)
        last_notice = time.monotonic()
        while process.poll() is None:
            time.sleep(2)
            if time.monotonic() - last_notice >= 30:
                print(f"Upload still running: {time.monotonic() - started:.0f}s; log {log_path.name}", flush=True)
                last_notice = time.monotonic()
    result_text = log_path.read_text(encoding="utf-8", errors="replace")
    print(result_text[-3500:], flush=True)
    if process.returncode != 0:
        raise RuntimeError(f"Kaggle upload failed; inspect {log_path}")
    for attempt in range(40):
        result = subprocess.run([cli, "datasets", "status", dataset, "--format", "json"],
                                capture_output=True, text=True)
        if result.returncode == 0:
            status = json.loads(result.stdout)
            if publication_ready(status, expected_version):
              

def verify_remote_data(root: Path) -> dict:
    """Compare all uploaded file names and sizes with the verified local ZIP."""
    from kaggle.api.kaggle_api_extended import KaggleApi
    import pandas as pd

    config = json.loads((root / "src/configs/kaggle.json").read_text())
    dataset = f"{config['owner']}/{config['data_slug']}"
    api = KaggleApi()
    api.authenticate()
    remote = {}
    token = None
    while True:
        page = api.dataset_list_files(dataset, page_token=token, page_size=200)
        remote.update({item.name: item.total_bytes for item in page.files})
        token = page.next_page_token
        if not token:
            break
    markers = [name for name in remote if name.endswith("caries/data.yaml")]
    if len(markers) != 1:
        raise RuntimeError("Cannot locate one expanded prepared dataset on Kaggle")
    prefix = markers[0][:-len("caries/data.yaml")]
    archive_path = root / ".venv/work/kaggle/staging/data/prepared.zip"
    with zipfile.ZipFile(archive_path) as archive:
        rows = [{"file": prefix + item.filename, "local_bytes": item.file_size,
                 "remote_bytes": remote.get(prefix + item.filename),
                 "matches": remote.get(prefix + item.filename) == item.file_size}
                for item in archive.infolist() if not item.is_dir()]
    report = root / "docs/reports/data_preparation"
    pd.DataFrame(rows).to_csv(report / "kaggle_remote_inventory.csv", index=False)
    result = {"dataset": dataset, "checked_files": len(rows), "remote_files": len(remote),
              "expanded_prefix": prefix, "mismatches": sum(not row["matches"] for row in rows),
              "verification": "Remote filenames and byte sizes; local ZIP separately checked by CRC and SHA-256"}
    write_json(report / "kaggle_remote_verification.json", result)
    if result["mismatches"]:
        raise RuntimeError("Remote inventory differs from the prepared archive; inspect the CSV report")
    return result

## 3. Build the data package

The package preserves two separate YOLO datasets. CRC and SHA-256 checks protect the generated archive; reruns reuse the same verified package.

In [3]:
data_stage = package_data(ROOT)
package_receipt = json.loads((data_stage / "package_receipt.json").read_text())
display(pd.DataFrame([package_receipt]))
display(pd.DataFrame([{"file": path.name, "bytes": path.stat().st_size}
                      for path in sorted(data_stage.iterdir()) if path.is_file()]))


,input_signature,archive_sha256,archive_bytes,archive_members
0,a713b4ccde1c389b1c1b7611df18455abed22a38bf0f9a65643c8f135ba7030f,8b7107a59c81e647bdfdd72bc5bf95806087cf3b686b684d09d21ff8e77fb2a7,1022817704,6492


,file,bytes
0,DATA_CARD.md,2401
1,dataset-metadata.json,153
2,package_receipt.json,242
3,prepared.zip,1022817704
4,README.md,2703
5,sources.json,781


## 4. Publish the public data dataset

Existing authentication is reused. No credential is embedded in the notebook. This cell performs the requested upload and waits for a verified ready/public state.

In [4]:
data_receipt = publish_stage(ROOT, "data", data_stage)
display(pd.DataFrame([data_receipt]))
remote_verification = verify_remote_data(ROOT)
display(pd.DataFrame([remote_verification]))


data: identical dataset version already uploaded; visibility verified


,dataset,url,version,private,status,artifact_sha256,artifact_bytes,mode,log
0,trihuynhviprovcl/dentalselfie-data-v1,https://www.kaggle.com/datasets/trihuynhviprovcl/dentalselfie-data-v1,1,False,ready,8b7107a59c81e647bdfdd72bc5bf95806087cf3b686b684d09d21ff8e77fb2a7,1022817704,create,.venv/work/kaggle/upload_logs/data_1792508001.log


,dataset,checked_files,remote_files,expanded_prefix,mismatches,verification
0,trihuynhviprovcl/dentalselfie-data-v1,6492,6496,prepared/,0,Remote filenames and byte sizes; local ZIP separately checked by CRC and SHA-256


## 5. Refresh the public code dataset

This publishes the current source layout and the completed data-processing notebooks, so the later training notebooks can use the matching code version.

## Public ROI and completed model evidence

The user explicitly authorized public research publication. Export only verified train/validation crops and collected v4 model evidence. Credentials and personal/pilot photos are excluded. No training is launched here.

In [5]:
from dentalselfie.public_artifacts import package_research
for kind in ("roi", "results"):
    stage = package_research(ROOT, kind)
    receipt = publish_stage(ROOT, kind, stage)
    display(pd.DataFrame([receipt]))

roi: identical dataset version already uploaded; visibility verified


,dataset,url,version,private,status,artifact_sha256,artifact_bytes,mode,log,recovery
0,trihuynhviprovcl/dentalselfie-gum-roi-v1,https://www.kaggle.com/datasets/trihuynhviprovcl/dentalselfie-gum-roi-v1,1,False,ready,7be1b280f2d30998328d5f3f9a8d258a5e1d8f83acae1780e4ff2b43ed6d05c0,240211396,create,.venv/work/kaggle/upload_logs/roi_1792557859.log,Recovered after search index lag; direct metadata verified and remote content_man...


Uploading results: 685.9 MB to trihuynhviprovcl/dentalselfie-yolo-v4-results (public)


Upload still running: 30s; log results_1792558517.log


Upload still running: 60s; log results_1792558517.log


Your public Dataset is being created. Please check progress at https://www.kaggle.com/datasets/trihuynhviprovcl/dentalselfie-yolo-v4-results



Waiting for Kaggle to finish processing results ...


Waiting for Kaggle to finish processing results ...


,dataset,url,version,private,status,artifact_sha256,artifact_bytes,mode,log
0,trihuynhviprovcl/dentalselfie-yolo-v4-results,https://www.kaggle.com/datasets/trihuynhviprovcl/dentalselfie-yolo-v4-results,1,False,ready,7041b714caaa4bb3bf27f949770c9c405914329c92f8c6e99782fd7f28c564e0,685933916,create,.venv/work/kaggle/upload_logs/results_1792558517.log


In [6]:
code_stage = package_code(ROOT)
code_receipt = publish_stage(ROOT, "code", code_stage)
display(pd.DataFrame([code_receipt]))


Uploading code: 22.8 MB to trihuynhviprovcl/dentalselfie-code (public)


Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/trihuynhviprovcl/dentalselfie-code



Waiting for Kaggle to finish processing code ...


,dataset,url,version,private,status,artifact_sha256,artifact_bytes,mode,log
0,trihuynhviprovcl/dentalselfie-code,https://www.kaggle.com/datasets/trihuynhviprovcl/dentalselfie-code,13,False,ready,52cc99e855858abdf364813f88d9d72165d032a4d3f627d99189e87953640eaa,22779797,version,.venv/work/kaggle/upload_logs/code_1792558627.log


## 6. Verified publication handoff

In [7]:
from IPython.display import Markdown

receipts = json.loads((REPORT / "kaggle_upload_receipt.json").read_text())
display(pd.DataFrame(receipts).T[["dataset", "version", "private", "status", "artifact_sha256"]])
expected_private = json.loads((ROOT / "src/configs/kaggle.json").read_text()).get("private", True)
assert all(item["private"] == expected_private and item["status"] == "ready" for item in receipts.values())
display(Markdown(f"""### Publication completed

- [Prepared data]({data_receipt['url']}), version **{data_receipt['version']}**, public and ready.
- [Matching code]({code_receipt['url']}), version **{code_receipt['version']}**, public and ready.
- **{len(manifests):,} prepared images** across two independent tasks.
- Local audit, split-lock and upload receipts are saved for reproduction.
- **No training job was launched.** Notebook 05 is executed; CNN/ResNet baseline implementation is the next stage."""))


,dataset,version,private,status,artifact_sha256
data,trihuynhviprovcl/dentalselfie-data-v1,1,False,ready,8b7107a59c81e647bdfdd72bc5bf95806087cf3b686b684d09d21ff8e77fb2a7
code,trihuynhviprovcl/dentalselfie-code,13,False,ready,52cc99e855858abdf364813f88d9d72165d032a4d3f627d99189e87953640eaa
roi,trihuynhviprovcl/dentalselfie-gum-roi-v1,1,False,ready,7be1b280f2d30998328d5f3f9a8d258a5e1d8f83acae1780e4ff2b43ed6d05c0
results,trihuynhviprovcl/dentalselfie-yolo-v4-results,1,False,ready,7041b714caaa4bb3bf27f949770c9c405914329c92f8c6e99782fd7f28c564e0


### Publication completed

- [Prepared data](https://www.kaggle.com/datasets/trihuynhviprovcl/dentalselfie-data-v1), version **1**, public and ready.
- [Matching code](https://www.kaggle.com/datasets/trihuynhviprovcl/dentalselfie-code), version **13**, public and ready.
- **3,242 prepared images** across two independent tasks.
- Local audit, split-lock and upload receipts are saved for reproduction.
- **No training job was launched.** Notebook 05 is executed; CNN/ResNet baseline implementation is the next stage.

## Original-source ROI publication - post-07 recovery study

Publish only the verified train/validation local crops from executed notebook 08. This independent publication does not rerun earlier dataset uploads or start GPU training. Source photographs, test crops, credentials and personal photos are excluded. Preserve ROI v1 as the old-preparation control.

In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/configs/kaggle.json').exists())
sys.path.insert(0, str(ROOT/'src'))
from dentalselfie.public_artifacts import package_research
from dentalselfie.publishing import publish_stage
raw_audit = json.loads((ROOT/'docs/reports/raw_roi_preparation/audit.json').read_text())
assert raw_audit['status']=='verified' and raw_audit['test_images_opened']==0
stage = package_research(ROOT, 'raw_roi')
print('Allowlisted raw-source crop archive:', stage)
raw_roi_publication = publish_stage(ROOT, 'raw_roi', stage)
print(json.dumps(raw_roi_publication, indent=2))

Allowlisted raw-source crop archive: C:\Users\huynh\Desktop\oral-disease\.venv\work\kaggle\staging\raw_roi


Uploading raw_roi: 149.2 MB to trihuynhviprovcl/dentalselfie-gum-raw-roi-v1 (public)


Upload still running: 30s; log raw_roi_1792589470.log


Your public Dataset is being created. Please check progress at https://www.kaggle.com/datasets/trihuynhviprovcl/dentalselfie-gum-raw-roi-v1



Waiting for Kaggle to finish processing raw_roi ...


Waiting for Kaggle to finish processing raw_roi ...


{
  "dataset": "trihuynhviprovcl/dentalselfie-gum-raw-roi-v1",
  "url": "https://www.kaggle.com/datasets/trihuynhviprovcl/dentalselfie-gum-raw-roi-v1",
  "version": 1,
  "private": false,
  "status": "ready",
  "artifact_sha256": "fb15c0b877079fb8a6fdb937f8f9768ccb5b809e9250d89919bee60429c04b8b",
  "artifact_bytes": 149185155,
  "mode": "create",
  "log": ".venv/work/kaggle/upload_logs/raw_roi_1792589470.log"
}


## Recovery-study code publication

Publish the executed notebook 08 and self-contained 09.x code after their preparation. This code-only dataset update does not start or replace GPU kernel runs. Existing public research visibility is preserved; no base-model weights or credentials are bundled.

In [1]:
from pathlib import Path
import sys, json
ROOT=next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/configs/kaggle.json').exists())
sys.path.insert(0,str(ROOT/'src'))
from dentalselfie.package import package_code
from dentalselfie.publishing import publish_stage
code_stage=package_code(ROOT)
code_publication=publish_stage(ROOT,'code',code_stage)
print(json.dumps(code_publication,indent=2))

Uploading code: 24.4 MB to trihuynhviprovcl/dentalselfie-code (public)


Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/trihuynhviprovcl/dentalselfie-code



{
  "dataset": "trihuynhviprovcl/dentalselfie-code",
  "url": "https://www.kaggle.com/datasets/trihuynhviprovcl/dentalselfie-code",
  "version": 14,
  "private": false,
  "status": "ready",
  "artifact_sha256": "fd4b8bf843a405a4f6d5539c1bcaaf8980f35c5670661d1bb20ec3d28b5daa00",
  "artifact_bytes": 24386375,
  "mode": "version",
  "log": ".venv/work/kaggle/upload_logs/code_1792590037.log"
}
